In [38]:
import pandas as pd

# Rutas relativas desde donde estás ejecutando el notebook
df_train = pd.read_csv("data/cleaned/train_clean.csv")
df_oot = pd.read_csv("data/cleaned/oot_clean.csv")

# Verifica que se cargaron bien
print("Train shape:", df_train.shape)
print("OOT shape:", df_oot.shape)
df_train.head()

Train shape: (41313, 58)
OOT shape: (10104, 57)


,ID_CORRELATIVO,CODMES_x,FLG_BANCARIZADO,RANG_INGRESO,FLAG_LIMA_PROVINCIA,EDAD,ANTIGUEDAD,ATTRITION,RANG_SDO_PASIVO_MENOS0,SDO_ACTIVO_MENOS0,...,FLG_SDO_OTSSFF_MENOS1,FLG_SDO_OTSSFF_MENOS2,FLG_SDO_OTSSFF_MENOS3,FLG_SDO_OTSSFF_MENOS4,FLG_SDO_OTSSFF_MENOS5,TIPO_REQUERIMIENTO2,DICTAMEN,CODMES_y,PRODUCTO_SERVICIO_2,SUBMOTIVO_2
0,35653,201208,1,Rang_ingreso_06,Lima,25.0,6.0,0,Rango_SDO_09,0,...,0,0,0,0,0,Reclamo,NO PROCEDE,201208,Producto 20,Submotivo 145
1,56800,201208,1,Rang_ingreso_01,Provincia,34.0,4.0,0,Rango_SDO_02,0,...,0,0,0,0,0,Reclamo,PROCEDE PARCIAL,202405,Producto 07,Submotivo 125
2,8410,201208,1,Rang_ingreso_04,Provincia,63.0,5.0,0,Rango_SDO_03,0,...,1,1,1,1,1,Solicitud,NO PROCEDE,202404,Producto 20,Submotivo 157
3,26877,201208,1,Rang_ingreso_06,Lima,26.0,7.0,0,Rango_SDO_07,0,...,1,0,1,1,1,Reclamo,NO PROCEDE,201208,Producto 17,Submotivo 31
4,26877,201208,1,Rang_ingreso_06,Lima,26.0,7.0,0,Rango_SDO_07,0,...,1,0,1,1,1,Reclamo,PROCEDE TOTAL,202407,Producto 07,Submotivo 140


In [39]:
df_oot.head()

,ID_CORRELATIVO,CODMES_x,FLG_BANCARIZADO,RANG_INGRESO,FLAG_LIMA_PROVINCIA,EDAD,ANTIGUEDAD,RANG_SDO_PASIVO_MENOS0,SDO_ACTIVO_MENOS0,SDO_ACTIVO_MENOS1,...,FLG_SDO_OTSSFF_MENOS1,FLG_SDO_OTSSFF_MENOS2,FLG_SDO_OTSSFF_MENOS3,FLG_SDO_OTSSFF_MENOS4,FLG_SDO_OTSSFF_MENOS5,TIPO_REQUERIMIENTO2,DICTAMEN,CODMES_y,PRODUCTO_SERVICIO_2,SUBMOTIVO_2
0,53259,202408,1,Rang_ingreso_03,Lima,29.0,8.0,Rango_SDO_01,0,0,...,1,1,1,1,1,Reclamo,PROCEDE TOTAL,202403,Producto 01,Submotivo 125
1,53104,202408,0,Desconocido,Lima,57.0,2.0,Rango_SDO_02,0,0,...,0,0,0,0,0,Reclamo,NO PROCEDE,202405,Producto 18,Submotivo 123
2,5720,202408,1,Rang_ingreso_03,Lima,28.0,1.0,Rango_SDO_04,0,0,...,0,0,0,0,0,Solicitud,PROCEDE TOTAL,202403,Producto 20,Submotivo 157
3,39512,202408,0,Rang_ingreso_01,Lima,23.0,3.0,Rango_SDO_01,0,0,...,0,0,0,0,0,Reclamo,PROCEDE TOTAL,202405,Producto 01,Submotivo 83
4,36445,202408,1,Rang_ingreso_01,Lima,27.0,2.0,Rango_SDO_06,0,0,...,1,1,1,1,1,Reclamo,NO PROCEDE,202407,Producto 20,Submotivo 125


In [40]:
# Ver distribución de clases relativas (%)
print("\nDistribución de clases (porcentaje):")
print(df_train["ATTRITION"].value_counts(normalize=True) * 100)



Distribución de clases (porcentaje):
ATTRITION
0    91.179532
1     8.820468
Name: proportion, dtype: float64


In [41]:
y = df_train["ATTRITION"]

In [42]:
# Elimina columnas no predictoras
columnas_a_excluir = ["ID_CORRELATIVO", "ATTRITION"]

X = df_train.drop(columns=columnas_a_excluir)


In [87]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import xgboost as xgb

# Escalar las variables importantes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)
y = y

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Antes de oversampling:", y_train.value_counts())

# Oversampling solo en train con SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Después de oversampling:", y_train_res.value_counts())

# Definir modelo XGBoost y parámetros para GridSearch
xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5],
    "learning_rate": [0.1, 0.3],
    "subsample": [0.8, 1]
}

grid_search = GridSearchCV(
    xgb_clf,
    param_grid,
    scoring="f1",
    cv=2,           # Reduce folds para que sea rápido
    n_jobs=-1,
    verbose=1
)

# Entrenar
grid_search.fit(X_train_res, y_train_res)

print(f"Mejores hiperparámetros: {grid_search.best_params_}")

# Evaluar en test
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))


Antes de oversampling: ATTRITION
0    30135
1     2915
Name: count, dtype: int64
Después de oversampling: ATTRITION
0    30135
1    30135
Name: count, dtype: int64
Fitting 2 folds for each of 16 candidates, totalling 32 fits


/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [04:06:46] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [04:06:46] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [04:06:46] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/conda/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [04:06:46] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329020674/work

Mejores hiperparámetros: {'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      7534
           1       0.49      0.29      0.36       729

    accuracy                           0.91      8263
   macro avg       0.71      0.63      0.66      8263
weighted avg       0.89      0.91      0.90      8263



In [95]:
# Instalación de librerías si no las tienes
# !pip install imblearn scikit-learn pandas

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

# 1. Cargar los datos
data = pd.read_csv("data/cleaned/train_clean.csv")

# 2. Preprocesamiento
cols_to_drop = ['ID_CORRELATIVO', 'CODMES_x', 'CODMES_y']
data.drop(columns=cols_to_drop, inplace=True)

# Codificar variables categóricas
le = LabelEncoder()
for col in data.select_dtypes(include=['object']).columns:
    data[col] = le.fit_transform(data[col].astype(str))

# 3. Definir variables predictoras y target
X = data.drop(columns='ATTRITION')
y = data['ATTRITION']

# 4. División en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Aplicar SMOTE al conjunto de entrenamiento
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 6. Entrenar Random Forest
model = RandomForestClassifier(random_state=42, n_estimators=200)
model.fit(X_train_smote, y_train_smote)

# 7. Evaluación
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Abandona', 'Abandona']))

print("\nROC-AUC Score:", roc_auc_score(y_test, y_proba))


Matriz de Confusión:
[[7283  251]
 [ 344  385]]

Reporte de Clasificación:
              precision    recall  f1-score   support

 No Abandona       0.95      0.97      0.96      7534
    Abandona       0.61      0.53      0.56       729

    accuracy                           0.93      8263
   macro avg       0.78      0.75      0.76      8263
weighted avg       0.92      0.93      0.93      8263


ROC-AUC Score: 0.8953203638703446


In [96]:
pip install mlflow


Note: you may need to restart the kernel to use updated packages.


In [98]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, recall_score
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn

# Definir el experimento
mlflow.set_experiment("Detección de Abandonos RF + SMOTE")

with mlflow.start_run():

    # 1. Cargar los datos
    data = pd.read_csv("data/cleaned/train_clean.csv")

    # 2. Preprocesamiento
    cols_to_drop = ['ID_CORRELATIVO', 'CODMES_x', 'CODMES_y']
    data.drop(columns=cols_to_drop, inplace=True)

    le = LabelEncoder()
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = le.fit_transform(data[col].astype(str))

    X = data.drop(columns='ATTRITION')
    y = data['ATTRITION']

    # 3. División
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 4. SMOTE
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

    # 5. Random Forest
    n_estimators = 200
    model = RandomForestClassifier(random_state=42, n_estimators=n_estimators)
    model.fit(X_train_smote, y_train_smote)

    # 6. Evaluación
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    print("Matriz de Confusión:")
    print(confusion_matrix(y_test, y_pred))

    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred, target_names=['No Abandona', 'Abandona']))

    print("\nROC-AUC Score:", auc)

    # 7. Registrar parámetros y métricas en MLflow
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("oversampling", "SMOTE")

    mlflow.log_metric("AUC", auc)
    mlflow.log_metric("F1_score", f1)
    mlflow.log_metric("Recall", recall)

    # 8. Registrar el modelo
    mlflow.sklearn.log_model(model, "modelo_random_forest_smote")


2025/06/27 05:11:04 INFO mlflow.tracking.fluent: Experiment with name 'Detección de Abandonos RF + SMOTE' does not exist. Creating a new experiment.


Matriz de Confusión:
[[7283  251]
 [ 344  385]]

Reporte de Clasificación:
              precision    recall  f1-score   support

 No Abandona       0.95      0.97      0.96      7534
    Abandona       0.61      0.53      0.56       729

    accuracy                           0.93      8263
   macro avg       0.78      0.75      0.76      8263
weighted avg       0.92      0.93      0.93      8263


ROC-AUC Score: 0.8953203638703446


2025/06/27 05:11:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [100]:
export MLFLOW_S3_ENDPOINT_URL=https://s3.us-east-2.amazonaws.com
export AWS_REGION=us-east-2

SyntaxError: invalid decimal literal (2933520031.py, line 1)